# E082 -- Pareto's Law: Global Inequality

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaulVanCode/protoscience-nasa-experiments/blob/main/notebooks/e082_pareto_inequality.ipynb)

**Objective:** Download Gini coefficient data from the World Bank API, analyze the global distribution of income inequality, identify trends by decade, compare countries, and compute Pareto alpha parameters.

The Gini coefficient (0 = perfect equality, 1 = one person has everything) is the standard measure of income inequality. Pareto's Law states that income follows a power-law distribution with exponent alpha related to the Gini coefficient.

## Data Source

- **Database:** World Bank Development Indicators
- **Indicator:** SI.POV.GINI (Gini index)
- **API:** `https://api.worldbank.org/v2/country/all/indicator/SI.POV.GINI`
- **Format:** JSON
- **Coverage:** ~170 countries, 1960-2024
- **License:** Creative Commons Attribution 4.0

In [ ]:
import urllib.request
import json
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Download Gini coefficient data from World Bank API
url = ("https://api.worldbank.org/v2/country/all/indicator/SI.POV.GINI"
       "?format=json&per_page=20000&date=1960:2024")
print("Downloading Gini coefficient data from World Bank...")
req = urllib.request.Request(url, headers={'User-Agent': 'ProtoScience/1.0'})
response = urllib.request.urlopen(req, timeout=120)
raw = json.loads(response.read().decode('utf-8'))

# World Bank API returns [metadata, data_array]
metadata = raw[0]
data_list = raw[1]
print(f"Total records: {metadata.get('total', '?')}")
print(f"Downloaded: {len(data_list)} records")

In [ ]:
# Parse: extract country, year, Gini value
records = []
for entry in data_list:
    if entry.get('value') is not None:
        try:
            gini = float(entry['value'])
            year = int(entry['date'])
            country = entry['country']['value']
            country_code = entry['countryiso3code']
            if 0 < gini <= 100 and 1960 <= year <= 2025:
                records.append({
                    'country': country,
                    'code': country_code,
                    'year': year,
                    'gini': gini
                })
        except (ValueError, TypeError, KeyError):
            continue

print(f"Valid Gini records: {len(records)}")

# Convert to arrays
countries = [r['country'] for r in records]
codes = [r['code'] for r in records]
years_arr = np.array([r['year'] for r in records])
ginis = np.array([r['gini'] for r in records])

unique_countries = list(set(countries))
unique_years = np.unique(years_arr)

print(f"Countries with data: {len(unique_countries)}")
print(f"Year range: [{unique_years.min()}, {unique_years.max()}]")
print(f"Gini range: [{ginis.min():.1f}, {ginis.max():.1f}]")

## Analysis

The relationship between the Gini coefficient G (on 0-1 scale) and the Pareto alpha is:

$$\alpha = \frac{1 + G^{-1}}{2} = \frac{G + 1}{2G}$$

For G = 0.30 (Scandinavia): alpha ~ 2.17 (steep distribution, more equal)  
For G = 0.60 (South Africa): alpha ~ 1.33 (fat tail, extreme inequality)

In [ ]:
# Compute Pareto alpha for each record
gini_decimal = ginis / 100.0  # Convert from 0-100 to 0-1 scale
pareto_alpha = (1 + 1.0 / gini_decimal) / 2.0

# Get most recent Gini for each country
latest = {}
for r in records:
    c = r['country']
    if c not in latest or r['year'] > latest[c]['year']:
        latest[c] = r

latest_ginis = [(v['country'], v['gini'], v['year']) for v in latest.values()]
latest_ginis.sort(key=lambda x: x[1])

print("=== Most Equal Countries (lowest Gini) ===")
for name, g, y in latest_ginis[:10]:
    alpha = (1 + 100.0 / g) / 2.0
    print(f"  {name:30s} Gini={g:.1f} ({y}) α={alpha:.2f}")

print(f"\n=== Most Unequal Countries (highest Gini) ===")
for name, g, y in latest_ginis[-10:]:
    alpha = (1 + 100.0 / g) / 2.0
    print(f"  {name:30s} Gini={g:.1f} ({y}) α={alpha:.2f}")

In [ ]:
# Trend by decade
decades = [(1960, 1969), (1970, 1979), (1980, 1989), (1990, 1999),
           (2000, 2009), (2010, 2019), (2020, 2029)]
decade_labels = [f"{d[0]}s" for d in decades]
decade_medians = []
decade_q25 = []
decade_q75 = []
decade_n = []

for d_start, d_end in decades:
    m = (years_arr >= d_start) & (years_arr <= d_end)
    if m.sum() > 0:
        decade_medians.append(np.median(ginis[m]))
        decade_q25.append(np.percentile(ginis[m], 25))
        decade_q75.append(np.percentile(ginis[m], 75))
        decade_n.append(m.sum())
    else:
        decade_medians.append(np.nan)
        decade_q25.append(np.nan)
        decade_q75.append(np.nan)
        decade_n.append(0)

print("\n=== Gini by Decade ===")
print(f"{'Decade':>8} {'Median':>8} {'IQR':>12} {'N':>6}")
print("-" * 40)
for label, med, q25, q75, n in zip(decade_labels, decade_medians, decade_q25, decade_q75, decade_n):
    if n > 0:
        print(f"{label:>8} {med:>8.1f} [{q25:.1f}-{q75:.1f}] {n:>6}")

In [ ]:
# === 4-SUBPLOT FIGURE ===
fig, axes = plt.subplots(2, 2, figsize=(16, 13))
fig.suptitle("E082: Pareto's Law — Global Income Inequality",
             fontsize=15, fontweight='bold')

# (a) Gini distribution histogram
ax = axes[0, 0]
ax.hist(ginis, bins=40, color='steelblue', edgecolor='black', linewidth=0.3, alpha=0.8)
ax.axvline(np.median(ginis), color='red', linestyle='--', linewidth=2,
           label=f'Median = {np.median(ginis):.1f}')
ax.set_xlabel('Gini Coefficient', fontsize=11)
ax.set_ylabel('Count (country-year observations)', fontsize=11)
ax.set_title(f'(a) Global Gini Distribution (N={len(ginis)})', fontsize=12)
ax.legend(fontsize=10)

# (b) Trend by decade (box-like plot)
ax = axes[0, 1]
valid_decades = [(l, m, q25, q75) for l, m, q25, q75, n in
                 zip(decade_labels, decade_medians, decade_q25, decade_q75, decade_n) if n > 0]
if valid_decades:
    labels, meds, q25s, q75s = zip(*valid_decades)
    x_pos = range(len(labels))
    ax.plot(x_pos, meds, 'ro-', markersize=8, linewidth=2, label='Median')
    ax.fill_between(x_pos, q25s, q75s, alpha=0.3, color='steelblue', label='IQR')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Gini Coefficient', fontsize=11)
ax.set_title('(b) Inequality Trend by Decade', fontsize=12)
ax.legend(fontsize=10)

# (c) Most recent Gini — top/bottom 15 countries
ax = axes[1, 0]
n_show = 12
bottom = latest_ginis[:n_show]  # most equal
top = latest_ginis[-n_show:]     # most unequal
combined = bottom + top
c_names = [f"{c[0][:18]}" for c in combined]
c_ginis = [c[1] for c in combined]
c_colors = ['steelblue'] * n_show + ['crimson'] * n_show
ax.barh(range(len(combined)), c_ginis, color=c_colors, edgecolor='black', linewidth=0.3)
ax.set_yticks(range(len(combined)))
ax.set_yticklabels(c_names, fontsize=7)
ax.set_xlabel('Gini Coefficient', fontsize=11)
ax.set_title(f'(c) Most/Least Equal Countries (latest data)', fontsize=12)
ax.invert_yaxis()

# (d) Pareto alpha distribution
ax = axes[1, 1]
valid_alpha = pareto_alpha[np.isfinite(pareto_alpha) & (pareto_alpha > 1) & (pareto_alpha < 10)]
ax.hist(valid_alpha, bins=40, color='coral', edgecolor='black', linewidth=0.3, alpha=0.8)
ax.axvline(np.median(valid_alpha), color='red', linestyle='--', linewidth=2,
           label=f'Median α = {np.median(valid_alpha):.2f}')
ax.axvline(2.0, color='green', linestyle=':', linewidth=2, alpha=0.7,
           label='α=2 (80/20 rule threshold)')
ax.set_xlabel('Pareto α', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('(d) Pareto Alpha Distribution', fontsize=12)
ax.legend(fontsize=10)
ax.annotate('Lower α = more inequality\nHigher α = more equality',
            xy=(0.6, 0.8), xycoords='axes fraction', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('e082_pareto_inequality.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nFigure saved: e082_pareto_inequality.png")

## Key Results Summary

| Analysis | Result |
|----------|--------|
| Countries with Gini data | ~170 in World Bank database |
| Global median Gini | ~35-40 (moderate inequality) |
| Most equal | Scandinavian/European countries (G ~ 25-30) |
| Most unequal | Southern African countries (G ~ 55-65) |
| Pareto alpha range | 1.3 (extreme inequality) to 2.5+ (egalitarian) |
| Decade trend | Variable — some convergence since 2000s |

**Physical interpretation:** Pareto's Law (1896) observed that income distributions follow a power law: P(income > x) ~ x^(-alpha). The Gini coefficient is a single-number summary of this distribution. The observed global range of Gini values (25-65) maps to Pareto alphas of ~1.3-2.5, with most countries clustered around alpha ~ 1.6-2.0. The famous "80/20 rule" (Pareto principle) corresponds to alpha ~ 1.16 (G ~ 80), which is more extreme than any current national Gini. Income inequality is a universal phenomenon, but its magnitude varies enormously across societies and time periods.